## 1) Introduction

Image captioning—automatically producing a natural-language description of a photograph—sits at the intersection of computer vision and natural language processing, making it one of the defining problems of multimodal machine learning. The task has direct real-world relevance: automatically generated alt text improves web accessibility for visually impaired users, and caption pipelines underpin image search, social media content moderation, and assistive robotics. Recent advances have brought together convolutional neural networks (CNNs), recurrent neural networks (RNNs), and large pretrained transformers to push captioning quality steadily upward on standard benchmarks (Hodosh et al., 2013; Wang et al., 2022).

This project implements and compares three progressively more capable architectures on the Flickr8k dataset to examine how each design choice—feature representation, language modeling strategy, and scale of pretraining—affects caption quality as measured by BLEU scores (Papineni et al., 2002). A non-generative retrieval baseline first establishes a lower bound. A generative CNN+LSTM encoder-decoder then tests whether a recurrent language model trained from scratch can surpass that bound. Finally, fine-tuning a pretrained vision-language transformer tests how much pretraining at scale changes the picture.

I am pursuing a career in computer vision and machine learning, and this project is a direct step in that direction. Transfer learning—taking a network pretrained on a large dataset and adapting it to a downstream task—is the foundation of modern large language models and vision-language models alike. Having come from a journalism background, I have seen first-hand how important accurate alt text is for inclusive communication, and I want to contribute to making its automation more viable.

The remainder of this report is organized as follows. Section 2 describes the Flickr8k dataset and all preprocessing steps. Section 3 explains the three ML architectures and the evaluation metric in detail. Section 4 presents and interprets the quantitative and qualitative results. Section 5 concludes.

#### The models

**Model 1 — ResNet-50 + KNN retrieval baseline.** A pretrained ResNet-50 CNN extracts a 2,048-dimensional feature vector for every image. At inference, a k-nearest-neighbors (KNN) search finds the most visually similar training image and returns one of its existing captions verbatim. This baseline demonstrates the limits of non-generative retrieval and sets a BLEU floor for the generative models.

**Model 2 — CNN+LSTM encoder-decoder.** The same frozen ResNet-50 encodes each image into a feature vector that initializes the hidden and cell states of an LSTM decoder, which is trained from scratch on the Flickr8k captions to generate a new caption word by word. This model combines the CNN and RNN material covered in class into an end-to-end pipeline.

**Model 3 — Fine-tuned vision-language transformer.** HuggingFace's `microsoft/git-base` (Generative Image-to-Text Transformer; Wang et al., 2022) is fine-tuned on the Flickr8k training set. Unlike the encoder-decoder, the transformer attends to all image patch tokens at every generation step rather than compressing the image into a single vector, and it enters fine-tuning already fluent from large-scale pretraining.

## 2) Data

#### Data source

The dataset used in this project is Flickr8k, originally released by the University of Illinois at Urbana-Champaign. The canonical reference is:

> Hodosh, M., Young, P., & Hockenmaier, J. (2013). Framing image description as a ranking task: Data, models and evaluation metrics. *Journal of Artificial Intelligence Research*, 47, 853–899.

The version used here was obtained from Kaggle (Jain, A. [adityajn105], *Flickr8k*, https://www.kaggle.com/datasets/adityajn105/flickr8k), which preserves the original UIUC release.

#### Dataset format and modalities

Flickr8k has two modalities: images and text. The downloaded archive contains an `Images/` folder and a `captions.txt` file.

**Images.** The folder holds 8,091 photographs. All are JPEG and RGB. Pixel dimensions vary: widths range from 164 px to 500 px (mean 455 px) and heights from 198 px to 500 px (mean 395 px). Most images are landscape-oriented. Because ResNet-50 requires a fixed 224 × 224 input, every image is resized before feature extraction.

**Captions.** `captions.txt` contains five independent human-written captions for each image, giving 40,455 rows in total. Every image has exactly five captions with no missing or corrupted entries. The average caption length is 10.81 words, ranging from 1 to 36 words; the distribution is moderately right-skewed but centered near the mean. The full vocabulary spans 8,441 unique words, of which 3,310 appear only once (hapax legomena). The most frequent content words are `dog`, `man`, `boy`, `woman`, and `girl`, reflecting the typical subjects of Flickr photographs.

#### Why this dataset

Flickr8k is a standard image captioning benchmark, so results can be contextualized against published numbers. It is large enough (8,091 images, 40,455 captions) that low performance cannot be attributed to data scarcity, yet small enough for three architectures to be trained end-to-end within the project timeline. The two-modality structure—photographs and free-form text—fulfills the multimedia project requirements and exercises both CNN and RNN tooling. The images are natural, human-photographed scenes with human-written descriptions, making any learned model directly applicable to real-world captioning tasks such as alt-text generation.

#### Preprocessing

**Train/validation/test split.** Images were split 75% train / 12.5% validation / 12.5% test (6,068 / 1,011 / 1,012 images) using a fixed random seed of 42 for reproducibility. The split was performed on unique image identifiers rather than on individual caption rows, so all five captions for a given image always remain in the same partition and no caption leaks across splits. All five reference captions per image are retained at evaluation time for BLEU scoring.

**Image preprocessing.** Every image was resized to 224 × 224 pixels to match the input expected by ResNet-50, then passed through Keras's `preprocess_input`, which performs the channel-wise mean subtraction used during ImageNet training (BGR channel order, per-channel means subtracted).

**Caption preprocessing.** Captions were lowercased and tokenized by retaining only alphabetic characters (`re.findall(r'[a-z]+', text.lower())`). For the CNN+LSTM model, `<START>` and `<END>` tokens were prepended and appended to each caption so the LSTM decoder learns when to begin and stop generating. Words appearing fewer than two times in the training set were replaced with `<UNK>`, reducing the decoder vocabulary from 7,468 training words to 4,506 tokens (including four special tokens). Sequences were zero-padded to the length of the longest training caption plus the two boundary tokens (38 positions total).

**No data cleaning was required.** All 8,091 image files were present on disk and cross-referenced correctly in the captions file, with no missing or corrupted entries.

## 3) Methods

#### ML task

The task is **image captioning**: given a single photograph as input, produce a fluent natural-language sentence describing its contents. This is a conditional text generation task in which the visual content of the image serves as the conditioning signal. It is not straightforwardly supervised classification or regression—the output space is a sequence of words from an open vocabulary—so standard tabular ML metrics (accuracy, MSE) are not applicable. Instead, captioning quality is measured by **BLEU**, a precision-based n-gram overlap metric described at the end of this section.

The three models span a progression from non-generative to generative, and from trained from scratch to pretrained at scale.

#### Model 1: ResNet-50 + KNN retrieval baseline

**ResNet-50 as a visual feature extractor.**
ResNet-50 is a 50-layer convolutional neural network pretrained on ImageNet's 1.28 million images across 1,000 object categories. CNNs are the standard tool for image data because they apply learned filters that slide across the spatial dimensions of the input, detecting local patterns (edges, textures, object parts) regardless of their position in the image. Stacking many convolutional layers lets the network build progressively higher-level representations.

ResNet-50 in particular uses residual connections—skip connections that add the input of a block directly to its output—which allow gradients to flow more easily during backpropagation and make it practical to train networks of this depth without degradation. Here, ResNet-50 is used only as a fixed feature extractor: the final classification head (`include_top=False`) is discarded, and global average pooling (`pooling='avg'`) compresses each image's final feature maps into a single 2,048-dimensional vector. The weights remain frozen at their ImageNet values throughout all three models, because ResNet-50 already learned general visual features that transfer well to Flickr8k's photographs.

**L2 normalization.**
Before the KNN search, all feature vectors are L2-normalized so that every vector has unit magnitude. Without normalization, Euclidean distance would be influenced by vector magnitude—a vector that is large simply because ResNet-50 happened to produce large activations for that image would appear far from other images purely due to scale, not visual dissimilarity. After L2 normalization, the squared Euclidean distance between two unit vectors is inversely proportional to their cosine similarity, so direction becomes the only measure of visual content and similarity comparisons are well-calibrated.

**k-Nearest Neighbors retrieval.**
A KNN index (scikit-learn `NearestNeighbors`, `k=1`, `metric='euclidean'`, brute-force search) is fitted on the 6,068 normalized training feature vectors. At inference, the same ResNet-50 extractor processes each test image into a normalized vector, and the KNN returns the single most similar training image. One of that training image's five captions is selected at random and returned as the predicted caption. Using a random draw from the five captions (rather than always returning the first) prevents two different test images that map to the same training neighbor from always generating identical output.

Because there are no trainable parameters and no hyperparameters to tune, the validation set serves no purpose for this model; evaluation is performed directly on the test set.

#### Model 2: CNN+LSTM encoder-decoder

**Why an LSTM.**
Caption generation is a sequential task: each word depends on the words that preceded it and on the image being described. RNNs are designed for sequential data because they maintain a hidden state that is updated at each time step, allowing information to propagate across the sequence. However, standard RNNs suffer from vanishing gradients: during backpropagation, gradients are multiplied by the recurrent weight matrix at each step, and if that matrix has singular values slightly below 1 the gradients shrink exponentially, so early time steps receive negligible gradient signal and the model cannot learn long-range dependencies.

An LSTM fixes this by maintaining two distinct memory streams: a cell state for long-term memory and a hidden state for short-term memory. At each time step, three learned gates (forget, input, and output) control how much information from each stream is retained, updated, or exposed. The model itself determines these gate values from the current input and previous state, so relevant information can be preserved across many steps without gradient vanishing.

**Architecture.**
The model follows a Functional API design with two input branches that converge in the LSTM decoder.

*Image encoder branch.* The frozen ResNet-50 extracts a 2,048-dimensional feature vector for each image (identical to Model 1). A Dense projection layer with ReLU activation maps this vector to 256 dimensions. The 256-dim output is then used as **both** the initial hidden state $h_0$ and the initial cell state $c_0$ of the LSTM decoder. This is how visual information enters the language model: the LSTM begins in a state that encodes the image content, then generates words from that starting point.

*Caption decoder branch.* Each caption token is looked up in a learned `Embedding` layer (vocabulary size 4,506 × embedding dimension 256), which maps each integer token index to a 256-dimensional dense vector trained jointly with the decoder. The `mask_zero=True` flag propagates a Boolean mask through the LSTM so that padding positions (index 0) do not influence the hidden state.

*LSTM decoder.* The LSTM has 256 units and `return_sequences=True`, so it outputs a 256-dim hidden vector at every time step. A final Dense layer with softmax over the 4,506-word vocabulary converts each hidden vector to a probability distribution over the next word.

Dropout (`rate=0.3`) is applied after the image projection, after the embedding, and after the LSTM output to reduce overfitting. Total trainable parameters: 3,361,434.

**Teacher forcing.**
Training uses teacher forcing: instead of feeding the model's own previous prediction at each step, the ground-truth token from the reference caption is provided. Specifically, for a caption of length $n$, the decoder input is $[\langle\text{START}\rangle, w_1, \ldots, w_{n-1}]$ and the target is $[w_1, \ldots, w_n, \langle\text{END}\rangle]$. This stabilizes early training by providing a consistent learning signal; without it, early prediction errors would compound across every subsequent time step and produce uninformative gradients.

Each image contributes five training samples (one per caption), giving 30,340 training sequence pairs.

**Loss function.**
A custom masked sparse categorical cross-entropy loss is used. Standard cross-entropy would average over all `SEQ_LEN` positions, including the padding positions that carry no information. The custom loss computes the per-position cross-entropy, zeroes out the padding positions using the mask derived from `tf.not_equal(y_true, 0)`, and divides the total by the count of real tokens only. This yields a more accurate gradient signal focused on the actual words.

**Training and early stopping.**
The model was trained with the Adam optimizer (default learning rate), batch size 64, and a maximum of 20 epochs. Early stopping monitors `val_loss` and halts training if it does not improve for 5 consecutive epochs (`patience=5`), then restores the best-epoch weights. Trained weights are cached to disk so that the extraction loop need not be repeated on subsequent runs.

**Inference.**
At test time, teacher forcing is replaced by **greedy decoding**: at each step the model selects the token with the highest predicted probability and appends it to the sequence, which is then fed back in at the next step. All 1,012 test images are decoded in a single batched loop—one forward pass per step rather than one full decoding pass per image—by processing the entire batch simultaneously through the model at each position.

#### Model 3: Fine-tuned vision-language transformer (microsoft/git-base)

**From transfer learning to fine-tuning.**
Models 1 and 2 both froze the ResNet-50 encoder and left its ImageNet weights unchanged. Model 3 extends transfer learning one step further: here the encoder *and* the decoder arrive pretrained—not on a classification task, but on a captioning task—and *all* parameters are fine-tuned together on Flickr8k with a small learning rate.

**GIT architecture.**
`microsoft/git-base` (Generative Image-to-Text Transformer; Wang et al., 2022) is a single-stream decoder-only transformer with 129 million parameters, pretrained on hundreds of millions of image-text pairs using a next-token prediction objective. It uses a Vision Transformer (ViT; Dosovitskiy et al., 2021) image encoder that divides the input image into a grid of 196 non-overlapping patches, projects each patch into a token embedding, and prepends a special classification token. These 197 visual tokens are concatenated with the text token embeddings to form the full input sequence, and the transformer decoder generates the caption by predicting each subsequent text token conditioned on all preceding text tokens *and* all 197 visual tokens simultaneously via self-attention.

This design eliminates both of the structural limitations of Model 2. First, there is no bottleneck: instead of compressing the entire image into a single 256-dim vector that initializes the LSTM once, every generation step attends directly to all 196 image patch tokens. Visual detail that is relevant to the current word can be retrieved at the moment it is needed. Second, self-attention connects any two positions in one step regardless of their distance in the sequence, so there is no chain of matrix multiplications through which gradients must travel—vanishing gradients across time are not an issue.

**Fine-tuning procedure.**
The HuggingFace `AutoProcessor` jointly preprocesses images and text in a single call, returning `pixel_values`, `input_ids`, and `attention_mask`. Labels are constructed by copying `input_ids` and setting padding positions to $-100$, the PyTorch convention for ignored positions in cross-entropy loss—functionally identical to the masked loss used in Model 2.

The full model was fine-tuned with the AdamW optimizer (learning rate $5 \times 10^{-5}$, weight decay 0.01), batch size 8 (small because all 129 million parameters receive gradient updates), and a maximum of 5 epochs. Early stopping with `patience=2` on validation loss restores the best-epoch checkpoint. As in Model 2, weights are cached to disk after training.

**Inference.**
Captions are generated via `model.generate()` using greedy decoding (`max_new_tokens=40`), matching the greedy strategy of Model 2. The processor's `batch_decode` with `skip_special_tokens=True` strips the beginning-of-sequence and end-of-sequence tokens, equivalent to Model 2 stripping `<START>`, `<END>`, and `<PAD>`.

#### Evaluation metric: BLEU

BLEU (Bilingual Evaluation Understudy) measures the n-gram precision of a generated caption against a set of reference captions. For each n-gram order $n$, BLEU counts how many n-grams in the prediction also appear in at least one reference, divides by the total number of n-grams in the prediction (clipping each n-gram count at its maximum count across the reference set to prevent repetition gaming), and then takes the geometric mean of the n-gram precisions across all orders up to $n$. A brevity penalty is applied when the prediction is shorter than the shortest reference, to prevent gaming the score with very short outputs.

For this project, corpus-level BLEU-1 through BLEU-4 are reported: scores are computed by pooling all n-gram counts across all 1,012 test images before computing one precision figure, rather than averaging per-image scores. BLEU-1 measures unigram (single-word) overlap; BLEU-4 requires matching four-word sequences and is therefore a much stricter measure of fluency and accuracy. Because higher-order n-gram precision can be zero for short or simple predictions (causing the geometric mean to collapse to zero), `SmoothingFunction().method1` from NLTK is applied for BLEU-2 through BLEU-4, which adds a small epsilon count to precisions that would otherwise be zero.

All three models are evaluated against the same 1,012 test images and the same five reference captions per image, using the same tokenization function (`re.findall(r'[a-z]+', text.lower())`), so any difference in BLEU reflects the model architecture alone.

## 4) Results and Discussion

#### Quantitative results

Table 1 summarizes the corpus-level BLEU scores on the 1,012-image test set for all three models.

| Model | BLEU-1 | BLEU-2 | BLEU-3 | BLEU-4 |
|-------|--------|--------|--------|--------|
| ResNet-50 + KNN (retrieval baseline) | 0.4257 | 0.2407 | 0.1353 | 0.0764 |
| CNN+LSTM encoder-decoder | 0.3976 | 0.2208 | 0.1179 | 0.0698 |
| GIT fine-tuned (`microsoft/git-base`) | — | — | — | — |

*Table 1. Corpus-level BLEU-1 through BLEU-4 on the held-out test set (1,012 images, 5 references each). GIT scores are to be filled in once the fine-tuning run completes.*

#### Model 1: Retrieval baseline

The retrieval baseline achieved BLEU-1 = 0.43 and BLEU-4 = 0.08. These scores reflect the fundamental limitation of a non-generative approach: the model cannot compose new language. It can only copy a caption that was written for a different image—one that happens to be visually similar by the ResNet-50 metric.

Qualitatively, the retrieved captions tend to capture the general subject of the query image (a person, a dog, a body of water) but miss details such as actions, colors, and background context. This is consistent with how ResNet-50 represents images: global average pooling over the final convolutional feature maps produces a 2,048-dimensional summary that encodes high-level category information well but discards spatial and fine-grained detail. The KNN consequently finds images of the same broad category but not necessarily the same specific scene.

BLEU-4 of 7.64% indicates that exact four-word sequence matches between predictions and references are infrequent, which is expected when the prediction is copied from an image that is only visually similar rather than visually identical.

#### Model 2: CNN+LSTM encoder-decoder

The CNN+LSTM model scores below the retrieval baseline on all four BLEU metrics, which is a surprising result for a generative model. The cause is mode collapse: greedy decoding produces the same caption for every test image regardless of the input. During training, the LSTM learned a single high-probability caption template—a generic description that minimizes cross-entropy loss across the full training distribution—and at inference the greedy argmax always selects this template.

The structural explanation is that the image conditions the LSTM exactly once, at initialization. After $h_0$ and $c_0$ are set from the projected image feature, each subsequent hidden state is computed from the previous hidden state and the most recently generated token alone. As the decoder processes more steps, the image signal is progressively diluted, and the model converges on the statistically dominant output. This is a known failure mode of single-injection encoder-decoders (Vinyals et al., 2015): injecting the image only at $h_0$ provides a weak conditioning signal that the LSTM can effectively ignore once it has learned the most probable caption pattern in the training data.

The closeness of the CNN+LSTM BLEU scores to the retrieval baseline—despite mode collapse—is partly explained by the retrieval model's own limitations: when all predictions are the same, the expected unigram overlap with five diverse references is still non-trivial if the predicted sentence contains common words (articles, prepositions, frequent nouns) that appear in almost every reference caption.

#### Model 3: GIT fine-tuned transformer

The GIT transformer is architecturally designed to avoid the mode collapse of Model 2. Because every generation step attends to all 196 image patch tokens via self-attention, the image conditioning signal remains present and accessible throughout generation rather than being injected once and diluted. Additionally, the model enters fine-tuning already pretrained on captioning at scale, so it does not need to learn basic English fluency from 30,340 training captions; it only needs to adapt its caption style to Flickr8k's vocabulary and phrasing.

The expected outcome is that GIT outperforms both earlier models, with the gap widening at BLEU-3 and BLEU-4 where multi-word phrase accuracy is required. The actual scores will be filled in the table above once the fine-tuning run completes.

## 5) Conclusion

This project compared three image captioning architectures on Flickr8k to trace the contribution of each major design decision. The ResNet-50 + KNN retrieval baseline demonstrated that visual similarity alone—without any language generation—can achieve BLEU-1 scores above 0.40, but is fundamentally limited by its inability to compose new descriptions. The CNN+LSTM encoder-decoder introduced a generative decoder, but its single-injection conditioning mechanism proved too weak to prevent mode collapse under greedy decoding, yielding scores marginally below the retrieval baseline. The fine-tuned GIT transformer addresses both shortcomings through persistent cross-attention to image patches and large-scale pretraining.

The progression illustrates several core ML principles covered in STAT 362, and many other concepts that I had to look up and research before applying: convolutional networks for local feature extraction from images, recurrent networks with gated memory (LSTMs) for sequence modeling, the practical consequences of vanishing gradients, teacher forcing as a training stabilization technique, masked loss functions to handle variable-length sequences, early stopping for regularization, and transfer learning as the dominant paradigm in modern deep learning. The transformer model further extends these ideas to attention-based architectures that have largely supplanted RNNs in sequence generation tasks.

## References

Jain, A. [adityajn105]. (n.d.). *Flickr8k* [Dataset]. Kaggle. https://www.kaggle.com/datasets/adityajn105/flickr8k

Dosovitskiy, A., Beyer, L., Kolesnikov, A., Weissenborn, D., Zhai, X., Unterthiner, T., ... & Houlsby, N. (2021). An image is worth 16x16 words: Transformers for image recognition at scale. *International Conference on Learning Representations (ICLR)*. https://arxiv.org/abs/2010.11929

Wang, P., Yang, A., Men, R., Lin, J., Shen, T., Li, X., ... & Zhou, J. (2022). GIT: A generative image-to-text transformer for vision and language. *Transactions on Machine Learning Research*. https://arxiv.org/abs/2205.14100

HuggingFace. (n.d.). *Image captioning* [Tutorial]. https://huggingface.co/docs/transformers/main/en/tasks/image_captioning

HuggingFace. (n.d.). *How to generate text*. https://huggingface.co/blog/how-to-generate

NLTK Project. (n.d.). `nltk.translate.bleu_score`. https://www.nltk.org/api/nltk.translate.bleu_score.html

scikit-learn. (n.d.). `sklearn.neighbors.NearestNeighbors`. https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html

Keras. (n.d.). *ResNet and ResNetV2*. https://keras.io/api/applications/resnet/

TensorFlow. (n.d.). `tf.keras.layers.LSTM`. https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM